In [2]:
!pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.8/100.2 MB 2.2 MB/s eta 0:00:45
    --------------------------------------- 1.6/100.2 MB 2.8 MB/s eta 0:00:36
    --------------------------------------- 2.1/100.2 MB 3.2 MB/s eta 0:00:31
   - -------------------------------------- 3.1/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 3.1/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 3.1/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 3.1/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 3.4/100.2 MB 1.8 MB/s eta 0:00:55
   - -------------------------------------- 4.2/100.2 MB 2.0 MB/s eta 0:00:49
   - -------------------------------------- 4.7/100.2 MB 2.1 MB/s eta 0:00:46
   - --

In [12]:
# ==============================
# IMPORTS
# ==============================
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==============================
# LOAD DATA
# ==============================
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
test_ids = test['ID']

train = train.drop(columns=['ID'])
test = test.drop(columns=['ID'])

# ==============================
# DATE FEATURES
# ==============================
for df in [train, test]:
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year'] = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day'] = df['deathdate'].dt.day
    df['weekday'] = df['deathdate'].dt.weekday
    # Cyclical month
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df.drop(columns=['deathdate','month'], inplace=True)

# ==============================
# AGE GROUP FEATURE
# ==============================
def age_group(age):
    if age < 5:
        return 'child'
    elif age < 18:
        return 'teen'
    elif age < 60:
        return 'adult'
    else:
        return 'elderly'

for df in [train, test]:
    df['age_group'] = df['age'].apply(age_group)

# ==============================
# NUMERIC INTERACTIONS
# ==============================
for df in [train, test]:
    df['temp_range'] = df['max_temperature'] - df['min_temperature']
    df['temp_mean_diff'] = df['avg_temperature'] - df['min_temperature']
    df['rain_temp'] = df['precipitation'] * df['avg_temperature']
    df['lat_long'] = df['latitude'] * df['longitude']
    # Weather flags
    df['hot_day'] = (df['avg_temperature'] > df['avg_temperature'].median()).astype(int)
    df['rainy_day'] = (df['precipitation'] > 0).astype(int)
    # Age-weather interactions
    df['child_hot'] = ((df['age_group']=='child') & (df['hot_day']==1)).astype(int)
    df['elderly_rain'] = ((df['age_group']=='elderly') & (df['rainy_day']==1)).astype(int)

# ==============================
# CATEGORICAL ENCODING
# ==============================
categorical_cols = ['zone', 'gender', 'location', 'age_group']

for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]], axis=0))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

# ==============================
# DATA PREP
# ==============================
X_train = train.drop(columns=['is_climate_sensitive'])
y_train = train['is_climate_sensitive']
X_test = test.copy()

X_train_lgb = X_train.astype(np.float32)
X_test_lgb = X_test.astype(np.float32)

# ==============================
# CROSS-VALIDATION
# ==============================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))
oof_preds = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"Fold {fold+1}")
    
    X_tr, X_val = X_train_lgb.iloc[tr_idx], X_train_lgb.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    
    # ---- LIGHTGBM ----
    lgb_model = LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=40,
        colsample_bytree=0.7,
        subsample=0.7,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1
    )
    lgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=100)
        ]
    )
    
    val_proba_lgb = lgb_model.predict_proba(X_val)[:,1]
    val_pred_lgb = (val_proba_lgb >= 0.5).astype(int)
    oof_preds[val_idx] = val_proba_lgb
    lgb_test_preds += lgb_model.predict_proba(X_test_lgb)[:,1] / skf.n_splits
    
    print("LightGBM F1:", f1_score(y_val, val_pred_lgb))
    print("LightGBM AUC:", roc_auc_score(y_val, val_proba_lgb))
    
    # ---- CATBOOST ----
    cat_model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        verbose=100,
        auto_class_weights='Balanced'
    )
    cat_model.fit(
        X_tr,
        y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100
    )
    
    val_proba_cat = cat_model.predict_proba(X_val)[:,1]
    val_pred_cat = (val_proba_cat >= 0.5).astype(int)
    cat_test_preds += cat_model.predict_proba(X_test)[:,1] / skf.n_splits
    
    print("CatBoost F1:", f1_score(y_val, val_pred_cat))
    print("CatBoost AUC:", roc_auc_score(y_val, val_proba_cat))

# ==============================
# ENSEMBLE (AVERAGE PROBABILITIES)
# ==============================
ensemble_test_preds = (lgb_test_preds + cat_test_preds) / 2

submission = pd.DataFrame({
    'ID': test_ids,
    'TargetF1': (ensemble_test_preds >= 0.5).astype(int),
    'TargetRAUC': ensemble_test_preds
})

submission.to_csv('submission_ensemble_upgraded.csv', index=False)
print("Upgraded ensemble submission saved!")

Fold 1
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000955 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2140
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.812716	valid_0's binary_logloss: 0.497717
Early stopping, best iteration is:
[19]	valid_0's auc: 0.830366	valid_0's binary_logloss: 0.54814
LightGBM F1: 0.8
LightGBM AUC: 0.8303658536585365
0:	test: 0.8279047	best: 0.8279047 (0)	total: 7.49ms	remaining: 15s
100:	test: 0.8331042	best: 0.8405432 (2)	total: 692ms	remaining: 13s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8405432373
bestIteration